Die benötigten Bibliotheken werden importiert

In [26]:
import simpy
import matplotlib.pyplot as plt
from datetime import date

Das Simpy Enviornment wird definiert

In [19]:
env = simpy.Environment()

Verbende Klasse (superclass) wird definiert

In [20]:
class Produktionsroboter:
    def __init__(self, id: str, energieverbrauch: float, kaufdatum: date, kapazitaet: int):
        self.id = id
        self.energieverbrauch = energieverbrauch
        self.kaufdatum = kaufdatum
        self.resource = simpy.Resource(env, capacity=kapazitaet)


    def request(self):
        """Einheitliche Zugriffsfunktion für SimPy-Prozesse"""
        return self.resource.request()



Erbende Klassen (subclass) wird definiert

In [21]:
class Maschine(Produktionsroboter):
    def __init__(self, id, energieverbrauch, kaufdatum, kapazitaet, produktionsgeschwindigkeit,
                 benoetigt_arbeitsroboter, produkt, produkt_name, rohstoff_eins, rohstoff_zwei=None):
        super().__init__(id, energieverbrauch, kaufdatum, kapazitaet)
        self.produktionsgeschwindigkeit: int = produktionsgeschwindigkeit
        self.benoetigt_arbeitsroboter: bool = benoetigt_arbeitsroboter

        self.produkt: simpy.resources = produkt
        self.produkt_name: str = produkt_name
        self.rohstoff_eins: simpy.resources = rohstoff_eins
        self.rohstoff_zwei: simpy.resources | None = rohstoff_zwei

    def print_details(self):
        print(f"Id: {self.id}")
        print(f"Energieverbrauch {self.energieverbrauch} kw/h")
        print(f"Produktionsgeschwindigkeit: {self.produktionsgeschwindigkeit} Einheiten/Minute")
        print(f"Wird ein Arbeitsroboter benötigt: {self.benoetigt_arbeitsroboter}")
        print("")


class Arbeitsroboter(Produktionsroboter):
    liste_erfuellte_arbeitsauftraege: list[str, int] | list

    def __init__(self, id, energieverbrauch, kaufdatum, kapazitaet, liste_erfuellte_arbeitsauftraege=[]):
        super().__init__(id, energieverbrauch, kaufdatum, kapazitaet)
        self.liste_erfuellte_arbeitsauftraege = liste_erfuellte_arbeitsauftraege

    def print_details(self):
        print(f"Id: {self.id}")
        print(f"Energieverbrauch {self.energieverbrauch} kw/h")
        print(f"Aufträge wurden fertiggestellt: {self.liste_erfuellte_arbeitsauftraege}")
        print("")

Ressourcen

In [22]:
# Materiallager (Stores)
mehl_store = simpy.Store(env)
wasser_store = simpy.Store(env)
kalte_schokolade_store = simpy.Store(env)

teig_store = simpy.Store(env)
heisse_schokolade_store = simpy.Store(env)
marzipan_store = simpy.Store(env)

for _ in range(5):
    mehl_store.put("Mehl")
    wasser_store.put("Wasser")
    kalte_schokolade_store.put("Kalte Schokolade")



Initialisieren von 5 Objekten (2x Arbeitsroboter; 3x Maschine)

In [23]:
arbeitsroboter_one = Arbeitsroboter("AR1", 0.5, date(2021, 3, 12), 1)
arbeitsroboter_two = Arbeitsroboter("AR2", 0.5, date(2021, 3, 12), 1)

arbeitsroboter_liste: list
arbeitsroboter_liste = [arbeitsroboter_one, arbeitsroboter_two]

teigmaschine = Maschine("TGM1", 60, date(2021, 3, 12), 1,  1, True, teig_store, "teig", wasser_store, mehl_store)
schokoladenaufbereitungsmaschine = Maschine("SAM1", 30, date(2019, 2, 1),1, 1, False, heisse_schokolade_store, "heiße_schokolade", kalte_schokolade_store)
ueberzugmaschine = Maschine("UZM1", 9, date(2021, 3, 12), 1, 1,  True, marzipan_store, "marzipan", teig_store, heisse_schokolade_store)

maschinen_liste = [teigmaschine, schokoladenaufbereitungsmaschine, ueberzugmaschine]

Die Simulationsprzoesse

In [24]:

def maschinen_prozess(env, maschine: Maschine):
    """Get: Objekt Maschine
    nimmt (get) die Materialien, welche für die Benutzung der Maschine benötigt werden, aus dem Rohstofflager.
    Anschließend produziert die Maschine über den Zeitraum maschine.produktionsgeschwindigkeit das Produkt und verändert
    den Store von maschine.produkt.
    """
    while True:
        # Auf Zutaten warten
        yield maschine.rohstoff_eins.get()

        if maschine.rohstoff_zwei is not None:
            yield maschine.rohstoff_zwei.get()

        if maschine.benoetigt_arbeitsroboter:
            roboter_gefunden = False
            while not roboter_gefunden:
                for roboter in arbeitsroboter_liste:
                    with roboter.resource.request() as req:
                        yield req
                        roboter_gefunden = True
                        print(f"[{env.now}] {maschine.id}: {roboter.id} ist zugewiesen.")
                        # Produktion starten
                        print(f"[{env.now}] {maschine.id}: Startet Produktion...")
                        maschine.last_start = env.now
                        yield env.timeout(maschine.produktionsgeschwindigkeit)
                        yield maschine.produkt.put(str(maschine.produkt_name))
                        print(f"[{env.now}] {maschine.id}: Produktion abgeschlossen mit {roboter.id}.")
                        roboter.liste_erfuellte_arbeitsauftraege.append(maschine.id)
                        break

                if not roboter_gefunden:
                    # Kein Roboter frei – warte kurz und versuche erneut
                    yield env.timeout(0.5)
        else:
            # --- Maschine ohne Roboter ---
            print(f"[{env.now}] {maschine.id}: Startet Teigproduktion...")
            maschine.last_start = env.now
            yield env.timeout(maschine.produktionsgeschwindigkeit)
            maschine.produkt.put(str(maschine.produkt_name))
            print(f"[{env.now}] {maschine.id}: Produktion abgeschlossen.")

Umgebung & Ressourcen

In [25]:
env.process(maschinen_prozess(env, teigmaschine))
env.process(maschinen_prozess(env, schokoladenaufbereitungsmaschine))
env.process(maschinen_prozess(env, ueberzugmaschine))

env.run(until=50)

[0] SAM1: Startet Teigproduktion...
[0] TGM1: AR1 ist zugewiesen.
[0] TGM1: Startet Produktion...
[1] SAM1: Produktion abgeschlossen.
[1] SAM1: Startet Teigproduktion...
[1] TGM1: Produktion abgeschlossen mit AR1.
[1] UZM1: AR1 ist zugewiesen.
[1] UZM1: Startet Produktion...
[2] SAM1: Produktion abgeschlossen.
[2] SAM1: Startet Teigproduktion...
[2] UZM1: Produktion abgeschlossen mit AR1.
[2] TGM1: AR1 ist zugewiesen.
[2] TGM1: Startet Produktion...
[3] SAM1: Produktion abgeschlossen.
[3] SAM1: Startet Teigproduktion...
[3] TGM1: Produktion abgeschlossen mit AR1.
[3] UZM1: AR1 ist zugewiesen.
[3] UZM1: Startet Produktion...
[4] SAM1: Produktion abgeschlossen.
[4] SAM1: Startet Teigproduktion...
[4] UZM1: Produktion abgeschlossen mit AR1.
[4] TGM1: AR1 ist zugewiesen.
[4] TGM1: Startet Produktion...
[5] SAM1: Produktion abgeschlossen.
[5] TGM1: Produktion abgeschlossen mit AR1.
[5] UZM1: AR1 ist zugewiesen.
[5] UZM1: Startet Produktion...
[6] UZM1: Produktion abgeschlossen mit AR1.
[6] 